# Qwen3-TTS on Google Colab

Run Qwen3-TTS with CUDA GPU acceleration on Google Colab.

**Requirements:** A Colab runtime with GPU (T4 or better).

**Setup:** Upload the `Qwen3-TTS_UserFiles` folder to Google Drive at `My Drive/Qwen3-TTS_UserFiles/`, then run all cells in order.

In [ ]:
# === Cell 1: Setup ===
import os, json

# Mount Google Drive (force_remount picks up newly synced files on re-run)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Link project from Drive to expected location
PROJECT_DIR = '/content/drive/My Drive/Qwen3-TTS_UserFiles'
HOME_DIR = os.path.expanduser('~/Qwen3-TTS_UserFiles')

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project not found at '{PROJECT_DIR}'.\n"
        "Upload the Qwen3-TTS_UserFiles folder to 'My Drive/Qwen3-TTS_UserFiles/' in Google Drive."
    )

if not os.path.exists(HOME_DIR):
    os.symlink(PROJECT_DIR, HOME_DIR)
    print(f'Linked {PROJECT_DIR} -> {HOME_DIR}')

# Create output directory (~/Downloads doesn't exist on Colab)
os.makedirs(os.path.expanduser('~/Downloads'), exist_ok=True)

# Install system dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null

# Install Python dependencies
!pip install -q torch>=2.0 qwen-tts "transformers==4.57.3" flask librosa soundfile numpy pydub requests gradio accelerate bitsandbytes scipy

# Configure for CUDA backend
config_path = os.path.expanduser('~/Qwen3-TTS_UserFiles/config.json')
with open(config_path) as f:
    config = json.load(f)

config['advanced']['backend'] = 'torch'
config['advanced']['dtype'] = 'float16'

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

# Verify GPU
import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\nSetup complete!')

In [ ]:
# === Cell 2: Start TTS Server ===
import subprocess, time, requests, sys

sys.path.insert(0, os.path.expanduser('~/Qwen3-TTS_UserFiles'))

# Kill any existing server on port 5123
!kill $(lsof -t -i:5123) 2>/dev/null || true

err_log = os.path.expanduser('~/server_stderr.log')
with open(err_log, 'w') as ef:
    server = subprocess.Popen(
        ['python', os.path.expanduser('~/Qwen3-TTS_UserFiles/voice_server.py')],
        stdout=subprocess.PIPE, stderr=ef
    )

# Wait for server to be ready (up to 90 seconds — first run downloads models)
server_url = 'http://127.0.0.1:5123'
print('Starting server...', end='')
for i in range(90):
    if server.poll() is not None:
        print(f'\nServer exited with code {server.returncode}')
        with open(err_log) as f:
            print(f.read()[-3000:])
        raise RuntimeError('Server failed to start — check errors above')
    try:
        resp = requests.get(f'{server_url}/health', timeout=1)
        if resp.status_code == 200:
            print(f'\nServer ready! (took {i+1}s)')
            health = resp.json()
            print(f'  Backend: {health.get("backend", "N/A")}')
            print(f'  Model size: {health.get("model_size", "N/A")}')
            break
    except requests.ConnectionError:
        pass
    print('.', end='', flush=True)
    time.sleep(1)
else:
    print('\nServer did not respond after 90s. Stderr:')
    with open(err_log) as f:
        print(f.read()[-3000:])

In [ ]:
# === Cell 3: Launch Gradio UI ===
# Access the UI via the public URL printed below.
from voice_ui import build_ui
demo = build_ui()
demo.launch(
    server_name='0.0.0.0',
    share=True,
    allowed_paths=[os.path.expanduser('~/Downloads'), '/tmp'],
)

In [ ]:
# === Cell 4: Quick Generation Example (without UI) ===
from voice_client import TTSClient

client = TTSClient()
output = client.generate(
    'Hello from Google Colab! This is Qwen3 TTS running on a GPU.',
    mode='design',
    description='A warm, friendly voice with clear articulation',
    output='colab_test.wav'
)
print(f'Generated: {output}')

# Play in notebook
from IPython.display import Audio
Audio(output)

In [ ]:
# === Cell 5: Troubleshooting ===
# Run this cell if you encounter errors to see the server log.
with open(os.path.expanduser('~/server_stderr.log')) as f:
    print(f.read()[-3000:])